# Notebook 5: Ablation Studies + Failure Mode Analysis + Statistical Analysis

## Part 1 — Ablation Studies
| Ablation | Question | What we vary |
|---|---|---|
| A1 | Does more training data help? | 10% / 50% / 100% corpus |
| A2 | Is the gain code-specific? | Test domain-tuned draft on math + general QA |
| A3 | Does gamma (lookahead) matter? | gamma = 1 / 3 / 5 / 7 / 10 |

## Part 2 — Failure Mode Analysis
| Analysis | Question |
|---|---|
| F1 | What error types cause failures? (SyntaxError, RuntimeError, WrongAnswer, Timeout) |
| F2 | Which problem categories fail most per condition? |
| F3 | What token types get rejected most in speculative decoding? |
| F4 | Do conditions fail on the same problems or different ones? |

## Part 3 — Statistical Analysis
| Analysis | Method |
|---|---|
| S1 | Bootstrap 95% confidence intervals for Pass@1 per condition |
| S2 | McNemar's test for pairwise Pass@1 differences (HumanEval + MBPP) |
| S3 | Bonferroni-corrected summary table with significance flags |

---
**Prerequisites:**
- Notebook 02: all 3 adapters trained (10%, 50%, 100%)
- Notebook 04: `benchmark_*.json` files saved to Drive

## Actual Runtimes (measured on A100)

| Section | Time |
|---|---|
| A1: Dataset size ablation (load 3 adapters + measure) | ~12 min |
| A2: Domain generalization (load 2 models + 3 domains) | ~10 min |
| A3: Gamma sweep (5 values, 8 prompts each) | ~8 min |
| F1: Error classification (re-execute failed completions) | ~5 min |
| F2: Category analysis (pure computation) | ~1 min |
| F3: Token rejection logging (2 models × 4 prompts) | ~6 min |
| F4: Failure overlap analysis (pure computation) | <1 min |
| S1: Bootstrap CI (B=2000 resamples × 4 conditions) | ~2 min |
| S2/S3: McNemar's test + Bonferroni correction | <1 min |
| **Total** | **~45 min** |

In [ ]:
!pip install -q "transformers==4.44.2" "peft==0.13.2" bitsandbytes accelerate datasets human-eval

In [ ]:
from google.colab import drive
import os, json, shutil

drive.mount('/content/drive', force_remount=True)
assert os.path.exists('/content/drive/MyDrive'), "Drive mount failed!"
print("Drive mounted.")

import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json, time, re, signal
from collections import defaultdict
from dataclasses import dataclass, field
from typing import List
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel, PeftConfig
from huggingface_hub import login

assert torch.cuda.is_available(), "Need GPU"
login()

TARGET_MODEL_ID = "codellama/CodeLlama-7b-hf"
DRAFT_MODEL_ID  = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

# All adapters from HuggingFace Hub
ADAPTER_10PCT  = "nishant-k/tinyllama-code-specdraft"
ADAPTER_50PCT  = "nishant-k/tinyllama-code-specdraft-50pct"
ADAPTER_100PCT = "nishant-k/tinyllama-code-specdraft-100pct"

VOCAB_SIZE_DRAFT = 32000   # TinyLlama vocab — mask CodeLlama tokens beyond this

RESULTS_DIR   = "./results"
DRIVE_RESULTS = "/content/drive/MyDrive/speculative-decoding-results"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(DRIVE_RESULTS, exist_ok=True)

# ── PEFT unknown-key patch — strips any unrecognised kwargs before __init__ ──
import inspect
from peft import config as _peft_config_module
from peft.config import PeftConfigMixin

if not getattr(PeftConfigMixin.from_peft_type, "_is_stripped_patch", False):
    _orig_from_peft_type = PeftConfigMixin.from_peft_type.__func__

    @classmethod
    def _patched_from_peft_type(cls, **kwargs):
        peft_type = kwargs.get("peft_type", None)
        try:
            config_cls = _peft_config_module.PEFT_TYPE_TO_CONFIG_MAPPING.get(peft_type, cls)
        except Exception:
            config_cls = cls
        valid = set(inspect.signature(config_cls.__init__).parameters.keys()) - {"self"}
        cleaned = {k: v for k, v in kwargs.items() if k in valid}
        dropped = set(kwargs) - set(cleaned)
        if dropped:
            print(f"  [PEFT patch] dropped unknown keys: {dropped}")
        return _orig_from_peft_type(cls, **cleaned)

    _patched_from_peft_type._is_stripped_patch = True
    PeftConfigMixin.from_peft_type = _patched_from_peft_type
    print("PEFT unknown-key patch applied.")
else:
    print("PEFT unknown-key patch already applied — skipping.")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(TARGET_MODEL_ID)
target    = AutoModelForCausalLM.from_pretrained(
    TARGET_MODEL_ID, torch_dtype=torch.float16, device_map="auto"
)
target.eval()
print(f"Target loaded | VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
@torch.no_grad()
def measure_acceptance_rate(prompts, draft_model, gamma=5, max_new_tokens=100, temperature=1.0):
    """Run speculative decoding on a list of prompts, return mean acceptance rate."""
    all_rates = []
    device    = next(target.parameters()).device
    for prompt in prompts:
        input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
        generated = input_ids.clone()
        total_draft, total_accepted, tokens_gen = 0, 0, 0

        while tokens_gen < max_new_tokens:
            draft_ids, draft_probs = [], []
            ctx = generated.clone()
            for _ in range(gamma):
                logits = draft_model(ctx).logits[:, -1, :] / temperature
                probs  = F.softmax(logits, dim=-1)
                token  = torch.multinomial(probs, 1)
                draft_ids.append(token)
                draft_probs.append(probs[0, token.item()].item())
                ctx = torch.cat([ctx, token], dim=-1)

            draft_seq = torch.cat(draft_ids, dim=-1)
            full_ctx  = torch.cat([generated, draft_seq], dim=-1)

            # Target forward with vocab masking
            tgt_logits_raw = target(full_ctx).logits[:, generated.shape[1]-1:-1, :] / temperature
            tgt_logits_raw[:, :, VOCAB_SIZE_DRAFT:] = float('-inf')
            tgt_probs = F.softmax(tgt_logits_raw, dim=-1)

            n_acc = 0
            for i in range(gamma):
                tok = draft_seq[0, i].item()
                p, q = tgt_probs[0, i, tok].item(), draft_probs[i]
                if torch.rand(1).item() <= min(1.0, p / (q + 1e-8)):
                    generated = torch.cat([generated, draft_seq[:, i:i+1]], dim=-1)
                    n_acc += 1; tokens_gen += 1
                    if tokens_gen >= max_new_tokens: break
                else:
                    tgt_last_raw = target(generated).logits[:, -1, :] / temperature
                    tgt_last_raw[:, VOCAB_SIZE_DRAFT:] = float('-inf')
                    tgt_last  = F.softmax(tgt_last_raw, dim=-1)[0]
                    corrected = F.relu(tgt_probs[0, i] - tgt_last)
                    mass = corrected.sum()
                    corrected = corrected / mass if mass > 1e-6 else tgt_probs[0, i]
                    generated = torch.cat([generated, torch.multinomial(corrected, 1).unsqueeze(0)], dim=-1)
                    tokens_gen += 1; break

            total_draft += gamma; total_accepted += n_acc
            if tokens_gen >= max_new_tokens: break

        all_rates.append(total_accepted / total_draft if total_draft > 0 else 0.0)
    return float(np.mean(all_rates))

---
## Part 1 — Ablation Studies
### A1 — Dataset Size vs Acceptance Rate

In [ ]:
CODE_PROMPTS = [
    "def fibonacci(n):\n    ", "def binary_search(arr, target):\n    ",
    "class Stack:\n    def __init__(self):\n        ", "def merge_sort(arr):\n    ",
    "def is_palindrome(s):\n    ", "import numpy as np\ndef normalize(x):\n    ",
    "def parse_json_file(path):\n    ", "def connect_to_db(host, port):\n    ",
]

ablation_a1 = {}
generic = AutoModelForCausalLM.from_pretrained(DRAFT_MODEL_ID, torch_dtype=torch.float16, device_map="auto")
generic.eval()
ablation_a1["generic (0%)"] = measure_acceptance_rate(CODE_PROMPTS, generic)
del generic; torch.cuda.empty_cache()

for label, path in [("10%", ADAPTER_10PCT), ("50%", ADAPTER_50PCT), ("100%", ADAPTER_100PCT)]:
    base  = AutoModelForCausalLM.from_pretrained(DRAFT_MODEL_ID, torch_dtype=torch.float16, device_map="auto")
    model = PeftModel.from_pretrained(base, path); model.eval()
    ablation_a1[label] = measure_acceptance_rate(CODE_PROMPTS, model)
    print(f"  {label}: {ablation_a1[label]:.3f}")
    del model; torch.cuda.empty_cache()

x_pos = [0, 10, 50, 100]
plt.figure(figsize=(8, 4))
plt.plot(x_pos, list(ablation_a1.values()), marker="o", linewidth=2, color="coral", markersize=8)
for x, y in zip(x_pos, ablation_a1.values()):
    plt.annotate(f"{y:.3f}", (x, y), textcoords="offset points", xytext=(5, 5))
plt.xlabel("Training Corpus Size (%)"); plt.ylabel("Mean Acceptance Rate")
plt.title("A1: Acceptance Rate vs Dataset Size")
plt.xticks(x_pos, ["0% (generic)", "10%", "50%", "100%"])
plt.grid(True, alpha=0.3); plt.tight_layout()
plt.savefig("ablation_a1_dataset_size.png", dpi=150); plt.show()

### A2 — Domain Generalization

In [ ]:
DOMAIN_PROMPTS = {
    "Code":       CODE_PROMPTS,
    "Math":       ["Solve for x: 2x + 5 = 13. Answer:", "What is the derivative of x^3 + 2x? Answer:",
                   "Calculate the area of a circle with radius 5. Answer:",
                   "If a train travels 60 mph for 2.5 hours, distance traveled is"],
    "General QA": ["The capital of France is", "Photosynthesis is the process by which",
                   "The first president of the United States was",
                   "Water boils at 100 degrees Celsius because"],
}

ablation_a2 = {}
generic_m = AutoModelForCausalLM.from_pretrained(DRAFT_MODEL_ID, torch_dtype=torch.float16, device_map="auto")
generic_m.eval()
base2 = AutoModelForCausalLM.from_pretrained(DRAFT_MODEL_ID, torch_dtype=torch.float16, device_map="auto")
tuned = PeftModel.from_pretrained(base2, ADAPTER_100PCT); tuned.eval()

for domain, prompts in DOMAIN_PROMPTS.items():
    rg = measure_acceptance_rate(prompts, generic_m)
    rt = measure_acceptance_rate(prompts, tuned)
    ablation_a2[domain] = {"generic": rg, "tuned": rt, "delta": rt - rg}
    print(f"{domain:12s}: generic={rg:.3f}  tuned={rt:.3f}  delta={rt-rg:+.3f}")

del generic_m, tuned; torch.cuda.empty_cache()

x = range(len(DOMAIN_PROMPTS)); w = 0.35
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar([i-w/2 for i in x], [ablation_a2[d]["generic"] for d in DOMAIN_PROMPTS], width=w, label="Generic", color="steelblue")
ax.bar([i+w/2 for i in x], [ablation_a2[d]["tuned"]   for d in DOMAIN_PROMPTS], width=w, label="Domain-Tuned", color="coral")
ax.set_xticks(list(x)); ax.set_xticklabels(list(DOMAIN_PROMPTS.keys()))
ax.set_ylabel("Mean Acceptance Rate"); ax.set_title("A2: Domain Generalization"); ax.legend()
plt.tight_layout(); plt.savefig("ablation_a2_domain_generalization.png", dpi=150); plt.show()

### A3 — Effect of Gamma

In [ ]:
GAMMA_VALUES = [1, 3, 5, 7, 10]
base3  = AutoModelForCausalLM.from_pretrained(DRAFT_MODEL_ID, torch_dtype=torch.float16, device_map="auto")
tuned3 = PeftModel.from_pretrained(base3, ADAPTER_100PCT); tuned3.eval()

ablation_a3 = {}
for gamma in GAMMA_VALUES:
    ablation_a3[gamma] = measure_acceptance_rate(CODE_PROMPTS, tuned3, gamma=gamma)
    print(f"  gamma={gamma}: {ablation_a3[gamma]:.3f}")
del tuned3; torch.cuda.empty_cache()

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(GAMMA_VALUES, [ablation_a3[g] for g in GAMMA_VALUES], marker="o", linewidth=2, color="mediumseagreen", markersize=8)
for g in GAMMA_VALUES:
    ax.annotate(f"{ablation_a3[g]:.3f}", (g, ablation_a3[g]), textcoords="offset points", xytext=(5, 5))
ax.set_xlabel("Gamma"); ax.set_ylabel("Mean Acceptance Rate")
ax.set_title("A3: Acceptance Rate vs Gamma"); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig("ablation_a3_gamma.png", dpi=150); plt.show()

---
## Part 2 — Failure Mode Analysis

Load benchmark results from Notebook 04 and answer:
1. **F1** — What error types cause failures?
2. **F2** — Which problem categories fail most?
3. **F3** — What token types get rejected most in speculative decoding?
4. **F4** — Do conditions fail on the same problems or different ones?

In [ ]:
import os
from datasets import load_dataset

CONDITIONS = ["baseline_generic", "domain_tuned", "medusa", "eagle2"]

# Load benchmark results saved by Notebook 04
# Try local results/ dir first, then Drive backup
benchmark = {}
for cond in CONDITIONS:
    for search_dir in [RESULTS_DIR, DRIVE_RESULTS]:
        path = os.path.join(search_dir, f"benchmark_{cond}.json")
        if os.path.exists(path):
            with open(path) as f:
                benchmark[cond] = json.load(f)
            print(f"Loaded {cond} from {search_dir}: {len(benchmark[cond]['humaneval']['results'])} HumanEval results")
            break
    if cond not in benchmark:
        print(f"WARNING: benchmark_{cond}.json not found — run Notebook 04 first")

# Load HumanEval problems (for prompts + test cases)
he_problems = {p["task_id"]: p for p in load_dataset("openai/openai_humaneval", split="test")}

### F1 — Error Type Classification
Re-execute failed completions to get the actual exception type.

In [ ]:
def classify_error(completion: str, problem: dict, timeout: float = 5.0) -> str:
    """Execute completion + tests and return error category."""
    def handler(signum, frame): raise TimeoutError()
    code = problem["prompt"] + completion + "\n\n" + problem["test"] + "\ncheck(" + problem["entry_point"] + ")"
    try:
        signal.signal(signal.SIGALRM, handler)
        signal.alarm(int(timeout))
        exec(compile(code, "<string>", "exec"), {})
        signal.alarm(0)
        return "pass"          # shouldn't reach here for failed cases
    except TimeoutError:
        return "Timeout"
    except SyntaxError:
        return "SyntaxError"
    except AssertionError:
        return "WrongAnswer"
    except (TypeError, ValueError, IndexError, KeyError, AttributeError) as e:
        return type(e).__name__
    except Exception as e:
        return type(e).__name__
    finally:
        signal.alarm(0)

# Classify errors for each condition (first failed sample per problem only)
error_counts = {}
for cond in CONDITIONS:
    if cond not in benchmark: continue
    counts = defaultdict(int)
    seen   = set()
    for r in benchmark[cond]["humaneval"]["results"]:
        if r["passed"] or r["task_id"] in seen:
            continue
        seen.add(r["task_id"])
        problem  = he_problems.get(r["task_id"])
        if problem:
            err = classify_error(r["completion"], problem)
            counts[err] += 1
    error_counts[cond] = dict(counts)
    print(f"{cond}: {dict(counts)}")

In [ ]:
all_error_types = sorted(set(e for c in error_counts.values() for e in c))
x   = np.arange(len(all_error_types))
w   = 0.2
colors = ["steelblue", "coral", "mediumseagreen", "mediumpurple"]

fig, ax = plt.subplots(figsize=(12, 5))
for i, (cond, color) in enumerate(zip(CONDITIONS, colors)):
    if cond not in error_counts: continue
    vals = [error_counts[cond].get(e, 0) for e in all_error_types]
    ax.bar(x + i*w, vals, width=w, label=cond, color=color)

ax.set_xticks(x + w * 1.5)
ax.set_xticklabels(all_error_types, rotation=20, ha="right")
ax.set_ylabel("Number of Failed Problems")
ax.set_title("F1: Error Type Distribution by Condition (HumanEval)")
ax.legend()
plt.tight_layout()
plt.savefig("failure_f1_error_types.png", dpi=150)
plt.show()

### F2 — Problem Category Analysis
Tag HumanEval problems by category and find which categories have the lowest pass rate.

In [ ]:
# Simple keyword-based categorizer for HumanEval prompts
CATEGORY_RULES = [
    ("String",     ["str", "string", "char", "substr", "palindrome", "anagram", "upper", "lower"]),
    ("List/Array", ["list", "array", "sort", "filter", "append", "index", "max", "min"]),
    ("Math",       ["sum", "product", "prime", "fibonacci", "factorial", "gcd", "lcm", "digit"]),
    ("Recursion",  ["recursive", "recursion", "tree", "binary", "depth"]),
    ("Dict/Set",   ["dict", "set", "map", "key", "value", "count", "frequency"]),
    ("Logic",      ["check", "valid", "detect", "find", "search", "match", "condition"]),
]

def categorize_problem(prompt: str) -> str:
    prompt_lower = prompt.lower()
    for category, keywords in CATEGORY_RULES:
        if any(kw in prompt_lower for kw in keywords):
            return category
    return "Other"

# Tag all problems
problem_categories = {tid: categorize_problem(p["prompt"]) for tid, p in he_problems.items()}

# Compute pass rate per category per condition
category_pass = {}
for cond in CONDITIONS:
    if cond not in benchmark: continue
    cat_results = defaultdict(list)
    # Use first sample per problem for pass rate
    seen = {}
    for r in benchmark[cond]["humaneval"]["results"]:
        if r["task_id"] not in seen:
            seen[r["task_id"]] = r["passed"]
    for tid, passed in seen.items():
        cat = problem_categories.get(tid, "Other")
        cat_results[cat].append(passed)
    category_pass[cond] = {cat: np.mean(v) for cat, v in cat_results.items()}

# Display
all_cats = sorted(set(c for d in category_pass.values() for c in d))
df_cat = pd.DataFrame(category_pass, index=all_cats).fillna(0)
print(df_cat.round(3).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(all_cats)); w = 0.2
for i, (cond, color) in enumerate(zip(CONDITIONS, colors)):
    if cond not in category_pass: continue
    vals = [category_pass[cond].get(c, 0) for c in all_cats]
    ax.bar(x + i*w, vals, width=w, label=cond, color=color)
ax.set_xticks(x + w*1.5); ax.set_xticklabels(all_cats, rotation=15, ha="right")
ax.set_ylabel("Pass Rate (Pass@1)")
ax.set_title("F2: Pass Rate by Problem Category")
ax.legend(); ax.set_ylim(0, 1.0)
plt.tight_layout()
plt.savefig("failure_f2_category_pass_rate.png", dpi=150)
plt.show()

### F3 — Token Rejection Analysis
For speculative decoding conditions, which token types (keywords, operators, identifiers, whitespace) get rejected most?
Run a fresh decoding pass with rejection logging enabled.

In [ ]:
import keyword

PYTHON_KEYWORDS = set(keyword.kwlist)

def classify_token(token_str: str) -> str:
    s = token_str.strip()
    if not s:                   return "Whitespace/Indent"
    if s in PYTHON_KEYWORDS:    return "Keyword"
    if s in {"(", ")", "[", "]", "{", "}", ":", ",", "."}: return "Punctuation"
    if s in {"+", "-", "*", "/", "==", "!=", "<", ">", "=", "+=", "-=", "**"}: return "Operator"
    if s.isdigit() or s.replace(".", "").isdigit(): return "Literal"
    if s.startswith(("'", '"')): return "String Literal"
    if s.startswith("#"):        return "Comment"
    return "Identifier"


@torch.no_grad()
def log_rejections(prompts, draft_model, gamma=5, max_new_tokens=80, temperature=1.0):
    """Run speculative decoding and log the token type of every rejected draft token."""
    rejected_types = defaultdict(int)
    accepted_types = defaultdict(int)
    device = next(target.parameters()).device

    for prompt in prompts:
        input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
        generated = input_ids.clone()
        tokens_gen = 0

        while tokens_gen < max_new_tokens:
            draft_ids, draft_probs = [], []
            ctx = generated.clone()
            for _ in range(gamma):
                logits = draft_model(ctx).logits[:, -1, :] / temperature
                probs  = F.softmax(logits, dim=-1)
                token  = torch.multinomial(probs, 1)
                draft_ids.append(token)
                draft_probs.append(probs[0, token.item()].item())
                ctx = torch.cat([ctx, token], dim=-1)

            draft_seq = torch.cat(draft_ids, dim=-1)
            full_ctx  = torch.cat([generated, draft_seq], dim=-1)

            tgt_logits_raw = target(full_ctx).logits[:, generated.shape[1]-1:-1, :] / temperature
            tgt_logits_raw[:, :, VOCAB_SIZE_DRAFT:] = float('-inf')
            tgt_probs = F.softmax(tgt_logits_raw, dim=-1)

            for i in range(gamma):
                tok     = draft_seq[0, i].item()
                tok_str = tokenizer.decode([tok])
                tok_cat = classify_token(tok_str)
                p, q    = tgt_probs[0, i, tok].item(), draft_probs[i]

                if torch.rand(1).item() <= min(1.0, p / (q + 1e-8)):
                    generated = torch.cat([generated, draft_seq[:, i:i+1]], dim=-1)
                    accepted_types[tok_cat] += 1
                    tokens_gen += 1
                    if tokens_gen >= max_new_tokens: break
                else:
                    rejected_types[tok_cat] += 1
                    tgt_last_raw = target(generated).logits[:, -1, :] / temperature
                    tgt_last_raw[:, VOCAB_SIZE_DRAFT:] = float('-inf')
                    tgt_last  = F.softmax(tgt_last_raw, dim=-1)[0]
                    corrected = F.relu(tgt_probs[0, i] - tgt_last)
                    mass = corrected.sum()
                    corrected = corrected / mass if mass > 1e-6 else tgt_probs[0, i]
                    generated = torch.cat([generated, torch.multinomial(corrected, 1).unsqueeze(0)], dim=-1)
                    tokens_gen += 1
                    break

            if tokens_gen >= max_new_tokens: break

    return dict(accepted_types), dict(rejected_types)


# Run for generic vs domain-tuned
rejection_analysis = {}
for label, adapter in [("generic", None), ("domain_tuned", ADAPTER_100PCT)]:
    base  = AutoModelForCausalLM.from_pretrained(DRAFT_MODEL_ID, torch_dtype=torch.float16, device_map="auto")
    model = PeftModel.from_pretrained(base, adapter) if adapter else base
    model.eval()
    acc, rej = log_rejections(CODE_PROMPTS[:4], model)
    rejection_analysis[label] = {"accepted": acc, "rejected": rej}
    total_rej = sum(rej.values())
    print(f"\n{label} — total rejections: {total_rej}")
    for cat, cnt in sorted(rej.items(), key=lambda x: -x[1]):
        rate = cnt / (cnt + acc.get(cat, 0) + 1e-8)
        print(f"  {cat:20s}: {cnt:4d} rejections ({rate*100:.1f}% rejection rate)")
    del model; torch.cuda.empty_cache()

In [ ]:
all_tok_cats = sorted(set(
    c for d in rejection_analysis.values() for c in d["rejected"]
))
x = np.arange(len(all_tok_cats)); w = 0.35

fig, ax = plt.subplots(figsize=(11, 5))
for i, (label, color) in enumerate([("generic", "steelblue"), ("domain_tuned", "coral")]):
    if label not in rejection_analysis: continue
    rej = rejection_analysis[label]["rejected"]
    acc = rejection_analysis[label]["accepted"]
    # Rejection rate per token type
    rates = [rej.get(c, 0) / (rej.get(c, 0) + acc.get(c, 0) + 1e-8) for c in all_tok_cats]
    ax.bar(x + i*w, rates, width=w, label=label, color=color)

ax.set_xticks(x + w/2); ax.set_xticklabels(all_tok_cats, rotation=20, ha="right")
ax.set_ylabel("Rejection Rate")
ax.set_title("F3: Token Rejection Rate by Token Type (Generic vs Domain-Tuned)")
ax.legend(); ax.set_ylim(0, 1.0)
plt.tight_layout()
plt.savefig("failure_f3_token_rejection.png", dpi=150)
plt.show()
print("Expected: domain-tuned should show lower rejection rates on Keywords and Identifiers")

### F4 — Do Conditions Fail on the Same Problems?
If conditions fail on *different* problems, an ensemble might help. If they fail on the *same* problems, the issue is fundamental difficulty.

In [ ]:
# Get the set of failed task_ids per condition (Pass@1, first sample)
failed_per_condition = {}
for cond in CONDITIONS:
    if cond not in benchmark: continue
    seen = {}
    for r in benchmark[cond]["humaneval"]["results"]:
        if r["task_id"] not in seen:
            seen[r["task_id"]] = r["passed"]
    failed_per_condition[cond] = {tid for tid, passed in seen.items() if not passed}
    print(f"{cond}: {len(failed_per_condition[cond])} failures")

# Overlap analysis
conds = [c for c in CONDITIONS if c in failed_per_condition]
print("\nFailure overlap (number of problems both conditions fail on):")
for i, c1 in enumerate(conds):
    for c2 in conds[i+1:]:
        overlap = failed_per_condition[c1] & failed_per_condition[c2]
        print(f"  {c1} ∩ {c2}: {len(overlap)} shared failures")

# Problems ONLY the baseline fails (domain-tuning fixes these)
if "baseline_generic" in failed_per_condition and "domain_tuned" in failed_per_condition:
    only_baseline = failed_per_condition["baseline_generic"] - failed_per_condition["domain_tuned"]
    only_tuned    = failed_per_condition["domain_tuned"] - failed_per_condition["baseline_generic"]
    print(f"\nProblems baseline fails but domain-tuned solves: {len(only_baseline)}")
    print(f"Problems domain-tuned fails but baseline solves : {len(only_tuned)}")

In [ ]:
# Venn-style overlap heatmap
n = len(conds)
overlap_matrix = np.zeros((n, n))
for i, c1 in enumerate(conds):
    for j, c2 in enumerate(conds):
        if c1 in failed_per_condition and c2 in failed_per_condition:
            overlap_matrix[i, j] = len(failed_per_condition[c1] & failed_per_condition[c2])

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(overlap_matrix, cmap="YlOrRd")
ax.set_xticks(range(n)); ax.set_xticklabels(conds, rotation=20, ha="right")
ax.set_yticks(range(n)); ax.set_yticklabels(conds)
for i in range(n):
    for j in range(n):
        ax.text(j, i, f"{overlap_matrix[i,j]:.0f}", ha="center", va="center", fontsize=11)
plt.colorbar(im)
ax.set_title("F4: Failure Overlap Matrix (shared failed problems)")
plt.tight_layout()
plt.savefig("failure_f4_overlap_matrix.png", dpi=150)
plt.show()

---
## Part 3 — Statistical Analysis

**Goal:** Quantify whether observed Pass@k differences between conditions are statistically significant,
or could be explained by sampling noise.

Three analyses:
- **S1** — Bootstrap 95% CI for Pass@1 and Pass@10 (HumanEval + MBPP)
- **S2** — McNemar's test for pairwise condition comparisons (per benchmark)
- **S3** — Bonferroni-corrected summary table (6 pairs × 2 benchmarks = 12 tests)

**Why McNemar's test?**
Each problem is a paired binary observation (did condition A pass? did condition B pass?).
McNemar's test is exactly designed for paired binary outcomes — it tests whether the
discordant pairs (A passes but B fails, or vice versa) are symmetric.

**Why bootstrap CIs?**
Pass@k is a non-linear statistic (it uses the unbiased estimator), so closed-form CIs
don't exist. Bootstrap resampling (problems with replacement) gives empirical CIs.

### S1 — Bootstrap Confidence Intervals for Pass@k

In [ ]:
from scipy import stats as scipy_stats

# ── Helpers ───────────────────────────────────────────────────────────────────

def pass_at_k_unbiased(n: int, c: int, k: int) -> float:
    """
    Unbiased estimator of Pass@k (Chen et al. 2021).
    n = samples per problem, c = number that pass, k = target k.
    Returns 0 if n < k.
    """
    if n < k:
        return 0.0
    if n - c < k:
        return 1.0
    from math import comb
    return 1.0 - comb(n - c, k) / comb(n, k)


def get_pass_vectors(benchmark_data: dict, suite: str = "humaneval") -> dict:
    """
    From benchmark JSON (condition → {humaneval: {results: [...]}, mbpp: {...}}),
    extract per-problem pass/fail vectors.

    Returns: {task_id: {"n": int, "c": int}}  (n samples, c passing)
    """
    results = benchmark_data.get(suite, {}).get("results", [])
    per_problem = defaultdict(lambda: {"n": 0, "c": 0})
    for r in results:
        tid = r["task_id"]
        per_problem[tid]["n"] += 1
        per_problem[tid]["c"] += int(r["passed"])
    return dict(per_problem)


def bootstrap_pass_at_k(
    pass_data: dict,
    k: int = 1,
    n_samples_per_problem: int = None,
    B: int = 2000,
    alpha: float = 0.05,
    rng_seed: int = 42,
) -> tuple:
    """
    Bootstrap 95% CI for Pass@k.

    pass_data: {task_id: {"n": int, "c": int}}
    Returns: (point_estimate, ci_low, ci_high)
    """
    rng      = np.random.default_rng(rng_seed)
    task_ids = list(pass_data.keys())
    N        = len(task_ids)
    n_per    = n_samples_per_problem or pass_data[task_ids[0]]["n"]

    # Point estimate
    point = np.mean([pass_at_k_unbiased(d["n"], d["c"], k) for d in pass_data.values()])

    # Bootstrap
    boot_scores = np.empty(B)
    for b in range(B):
        idx     = rng.integers(0, N, size=N)           # sample problems with replacement
        sampled = [pass_data[task_ids[i]] for i in idx]
        boot_scores[b] = np.mean([
            pass_at_k_unbiased(d["n"], d["c"], k) for d in sampled
        ])

    ci_low  = float(np.percentile(boot_scores, 100 * alpha / 2))
    ci_high = float(np.percentile(boot_scores, 100 * (1 - alpha / 2)))
    return float(point), ci_low, ci_high


# ── S1: Run bootstrap CIs for all conditions × benchmarks × k values ──────────

K_VALUES = [1, 10]
SUITES   = ["humaneval", "mbpp"]

# Load MBPP results if present (same structure as humaneval in benchmark JSON)
stat_results = {}   # {cond: {suite: {k: (point, lo, hi)}}}

for cond in CONDITIONS:
    if cond not in benchmark:
        print(f"SKIP {cond} — no benchmark data")
        continue
    stat_results[cond] = {}
    for suite in SUITES:
        data = get_pass_vectors(benchmark[cond], suite=suite)
        if not data:
            print(f"  {cond}/{suite}: no data")
            continue
        stat_results[cond][suite] = {}
        for k in K_VALUES:
            pt, lo, hi = bootstrap_pass_at_k(data, k=k, B=2000)
            stat_results[cond][suite][k] = (pt, lo, hi)
            print(f"  {cond:20s} | {suite:10s} | Pass@{k}: {pt:.3f}  95% CI [{lo:.3f}, {hi:.3f}]")

In [ ]:
# Plot bootstrap CIs — one subplot per benchmark, showing Pass@1 with error bars
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors_map = {
    "baseline_generic": "steelblue",
    "domain_tuned":     "coral",
    "medusa":           "mediumseagreen",
    "eagle2":           "mediumpurple",
}

for ax, suite in zip(axes, SUITES):
    available_conds = [c for c in CONDITIONS if c in stat_results and suite in stat_results[c]]
    x     = np.arange(len(available_conds))
    pts   = [stat_results[c][suite][1][0] for c in available_conds]
    lows  = [stat_results[c][suite][1][1] for c in available_conds]
    highs = [stat_results[c][suite][1][2] for c in available_conds]
    yerr_lo = [p - l for p, l in zip(pts, lows)]
    yerr_hi = [h - p for p, h in zip(pts, highs)]

    bar_colors = [colors_map.get(c, "gray") for c in available_conds]
    bars = ax.bar(x, pts, color=bar_colors, width=0.5, alpha=0.85)
    ax.errorbar(x, pts, yerr=[yerr_lo, yerr_hi], fmt="none",
                ecolor="black", elinewidth=1.5, capsize=6)

    for xi, (p, lo, hi) in enumerate(zip(pts, lows, highs)):
        ax.text(xi, hi + 0.01, f"{p:.3f}", ha="center", va="bottom", fontsize=9)

    ax.set_xticks(x)
    ax.set_xticklabels(available_conds, rotation=12, ha="right")
    ax.set_ylabel("Pass@1")
    ax.set_title(f"S1: Pass@1 with 95% Bootstrap CI\n({suite.upper()})")
    ax.set_ylim(0, 1.0)
    ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("stat_s1_bootstrap_ci.png", dpi=150)
plt.show()
print("Error bars = 95% bootstrap CI (B=2000 resamples)")

### S2 — McNemar's Test for Pairwise Significance

For each pair of conditions (6 pairs = C(4,2)), on each benchmark (HumanEval + MBPP),
build a 2×2 contingency table from per-problem Pass@1 outcomes and apply McNemar's test.

**Contingency table:**
```
                   Cond B: PASS    Cond B: FAIL
Cond A: PASS         n11               n10
Cond A: FAIL         n01               n00
```
- McNemar statistic with continuity correction: χ² = (|n10 − n01| − 1)² / (n10 + n01)
- df=1, so p = 1 − CDF(χ², 1)
- If n10 + n01 < 10: use exact binomial test instead (sparse discordant pairs)

In [ ]:
from itertools import combinations

def get_per_problem_pass1(benchmark_data: dict, suite: str = "humaneval") -> dict:
    """
    Extract per-problem Pass@1 (first sample only) as binary dict.
    Returns {task_id: bool}
    """
    results = benchmark_data.get(suite, {}).get("results", [])
    seen = {}
    for r in results:
        if r["task_id"] not in seen:
            seen[r["task_id"]] = bool(r["passed"])
    return seen


def mcnemar_test(pass_a: dict, pass_b: dict) -> dict:
    """
    McNemar's test comparing two conditions on the same set of problems.

    pass_a, pass_b: {task_id: bool}
    Returns dict with n11, n10, n01, n00, statistic, p_value, test_type
    """
    shared = sorted(set(pass_a) & set(pass_b))
    if not shared:
        return {"error": "no shared problems"}

    n11 = sum(1 for t in shared if     pass_a[t] and     pass_b[t])
    n10 = sum(1 for t in shared if     pass_a[t] and not pass_b[t])
    n01 = sum(1 for t in shared if not pass_a[t] and     pass_b[t])
    n00 = sum(1 for t in shared if not pass_a[t] and not pass_b[t])

    discordant = n10 + n01
    if discordant == 0:
        return {
            "n11": n11, "n10": n10, "n01": n01, "n00": n00,
            "statistic": 0.0, "p_value": 1.0,
            "test_type": "mcnemar", "note": "no discordant pairs",
        }

    if discordant < 10:
        # Exact binomial: under H0, P(B=n10 | n10+n01, 0.5)
        # Two-sided p-value
        result = scipy_stats.binomtest(n10, discordant, p=0.5, alternative="two-sided")
        return {
            "n11": n11, "n10": n10, "n01": n01, "n00": n00,
            "statistic": float(result.statistic),
            "p_value":   float(result.pvalue),
            "test_type": "exact_binomial",
        }

    # McNemar with continuity correction (Fleiss et al.)
    chi2 = (abs(n10 - n01) - 1) ** 2 / discordant
    p    = 1.0 - scipy_stats.chi2.cdf(chi2, df=1)
    return {
        "n11": n11, "n10": n10, "n01": n01, "n00": n00,
        "statistic": float(chi2),
        "p_value":   float(p),
        "test_type": "mcnemar_corrected",
    }


# ── Run all pairwise tests ────────────────────────────────────────────────────
mcnemar_rows = []
available_conds = [c for c in CONDITIONS if c in benchmark]

for suite in SUITES:
    for cond_a, cond_b in combinations(available_conds, 2):
        pa = get_per_problem_pass1(benchmark[cond_a], suite)
        pb = get_per_problem_pass1(benchmark[cond_b], suite)
        res = mcnemar_test(pa, pb)

        pt_a = np.mean(list(pa.values())) if pa else 0.0
        pt_b = np.mean(list(pb.values())) if pb else 0.0

        row = {
            "suite":   suite,
            "cond_A":  cond_a,
            "cond_B":  cond_b,
            "pass1_A": pt_a,
            "pass1_B": pt_b,
            "delta":   pt_a - pt_b,
            "n10":     res.get("n10", 0),
            "n01":     res.get("n01", 0),
            "statistic": res.get("statistic", 0.0),
            "p_raw":   res.get("p_value", 1.0),
            "test":    res.get("test_type", "?"),
        }
        mcnemar_rows.append(row)

df_mcnemar = pd.DataFrame(mcnemar_rows)
print(df_mcnemar[[
    "suite", "cond_A", "cond_B", "pass1_A", "pass1_B", "delta",
    "n10", "n01", "p_raw", "test"
]].round(4).to_string(index=False))

### S3 — Bonferroni-Corrected Summary Table

We run **12 tests** (6 pairs × 2 benchmarks).
Bonferroni correction: α_corrected = 0.05 / 12 ≈ 0.0042.
A result is marked significant (*) only if p_raw < α_corrected.

In [ ]:
N_TESTS        = len(df_mcnemar)           # 12 if all 4 conditions have data
ALPHA_FAMILY   = 0.05
ALPHA_BONF     = ALPHA_FAMILY / N_TESTS    # Bonferroni threshold

df_mcnemar["p_bonf_adj"] = df_mcnemar["p_raw"] * N_TESTS   # adjusted p (cap at 1.0)
df_mcnemar["p_bonf_adj"] = df_mcnemar["p_bonf_adj"].clip(upper=1.0)
df_mcnemar["sig"]        = df_mcnemar["p_raw"] < ALPHA_BONF
df_mcnemar["marker"]     = df_mcnemar["sig"].map({True: "* (sig)", False: "n.s."})

print(f"Bonferroni threshold: α = {ALPHA_FAMILY} / {N_TESTS} = {ALPHA_BONF:.4f}\n")

display_cols = ["suite", "cond_A", "cond_B", "delta", "p_raw", "p_bonf_adj", "marker"]
print(df_mcnemar[display_cols].round(4).to_string(index=False))

# ── Summary stats ─────────────────────────────────────────────────────────────
n_sig = df_mcnemar["sig"].sum()
print(f"\n{n_sig}/{N_TESTS} pairwise comparisons are significant at Bonferroni-corrected α={ALPHA_BONF:.4f}")

# Show which pairs are significant
sig_rows = df_mcnemar[df_mcnemar["sig"]]
if len(sig_rows) > 0:
    print("\nSignificant pairs:")
    for _, row in sig_rows.iterrows():
        direction = f"{row['cond_A']} > {row['cond_B']}" if row["delta"] > 0 else f"{row['cond_B']} > {row['cond_A']}"
        print(f"  [{row['suite']}] {direction}  |  Δ={row['delta']:+.3f}  p={row['p_raw']:.4f}")
else:
    print("\nNo pairs are significant at the corrected threshold.")

In [ ]:
# Heatmap of raw p-values (log scale) — makes it easy to see which pairs are far below threshold
pivot_he   = df_mcnemar[df_mcnemar["suite"] == "humaneval"].pivot(index="cond_A", columns="cond_B", values="p_raw")
pivot_mbpp = df_mcnemar[df_mcnemar["suite"] == "mbpp"].pivot(index="cond_A",  columns="cond_B", values="p_raw")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, pivot, title in zip(
    axes,
    [pivot_he, pivot_mbpp],
    ["HumanEval — McNemar p-values", "MBPP — McNemar p-values"]
):
    # Fill symmetric: use min(p(A,B), p(B,A)) so both triangles show the same value
    full = pivot.combine_first(pivot.T)
    np.fill_diagonal(full.values, np.nan)   # diagonal is undefined

    im = ax.imshow(
        np.log10(full.values.astype(float) + 1e-10),
        cmap="RdYlGn", vmin=-4, vmax=0
    )
    labels = list(full.columns)
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=15, ha="right")
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
    for i in range(len(labels)):
        for j in range(len(labels)):
            val = full.values[i, j]
            if not np.isnan(val):
                sig_marker = "*" if val < ALPHA_BONF else ""
                ax.text(j, i, f"{val:.3f}{sig_marker}", ha="center", va="center", fontsize=9)
    plt.colorbar(im, ax=ax, label="log10(p)")
    ax.set_title(f"S2/S3: {title}\n(* = significant at Bonferroni α={ALPHA_BONF:.4f})")

plt.tight_layout()
plt.savefig("stat_s2s3_mcnemar_pvalues.png", dpi=150)
plt.show()
print("Green = low p-value (significant), Red = high p-value (not significant)")

## Save All Results

In [ ]:
all_results = {
    "ablations": {
        "A1_dataset_size":          ablation_a1,
        "A2_domain_generalization": ablation_a2,
        "A3_gamma":                 ablation_a3,
    },
    "failure_modes": {
        "F1_error_types":    error_counts,
        "F2_category_pass":  category_pass,
        "F3_token_rejection": rejection_analysis,
        "F4_failure_overlap": {
            f"{c1}_and_{c2}": len(failed_per_condition.get(c1, set()) & failed_per_condition.get(c2, set()))
            for i, c1 in enumerate(conds) for c2 in conds[i+1:]
        },
    },
    "statistical_analysis": {
        "S1_bootstrap_ci": {
            cond: {
                suite: {
                    f"pass_at_{k}": {
                        "point":   stat_results[cond][suite][k][0],
                        "ci_low":  stat_results[cond][suite][k][1],
                        "ci_high": stat_results[cond][suite][k][2],
                    }
                    for k in K_VALUES
                    if k in stat_results[cond].get(suite, {})
                }
                for suite in SUITES
                if suite in stat_results.get(cond, {})
            }
            for cond in CONDITIONS
            if cond in stat_results
        },
        "S2_S3_mcnemar": df_mcnemar[[
            "suite", "cond_A", "cond_B", "delta",
            "n10", "n01", "p_raw", "p_bonf_adj", "marker"
        ]].round(4).to_dict(orient="records"),
        "bonferroni_threshold": float(ALPHA_BONF),
        "n_tests":              int(N_TESTS),
        "n_significant":        int(n_sig),
    },
}

out_path = os.path.join(RESULTS_DIR, "ablations_and_failure_modes.json")
with open(out_path, "w") as f:
    json.dump(all_results, f, indent=2, default=str)
shutil.copy(out_path, DRIVE_RESULTS)

print("Saved ablations_and_failure_modes.json → local + Drive")
print("\nSaved plots:")
for fname in [
    "ablation_a1_dataset_size.png", "ablation_a2_domain_generalization.png",
    "ablation_a3_gamma.png", "failure_f1_error_types.png",
    "failure_f2_category_pass_rate.png", "failure_f3_token_rejection.png",
    "failure_f4_overlap_matrix.png",
    "stat_s1_bootstrap_ci.png", "stat_s2s3_mcnemar_pvalues.png",
]:
    if os.path.exists(fname):
        shutil.copy(fname, DRIVE_RESULTS)
    print(f"  {fname}")